<a href="https://colab.research.google.com/github/Sowshik1206/Belarus-car-Price-Prediction/blob/main/caption_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!wget -q https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip
!wget -q https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip

!unzip -qq Flickr8k_Dataset.zip
!unzip -qq Flickr8k_text.zip

print("✅ DATASET READY")

^C
replace Flicker8k_Dataset/1000268201_693b08cb0e.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: replace CrowdFlowerAnnotations.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: ✅ DATASET READY


In [1]:
import os
print(os.listdir())

['.config', 'Flickr8k_Dataset.zip.1', '__MACOSX', 'Flickr_8k.trainImages.txt', 'Flickr8k.token.txt', 'CrowdFlowerAnnotations.txt', 'ExpertAnnotations.txt', 'readme.txt', 'Flickr8k_text.zip', 'Flickr_8k.devImages.txt', 'Flickr8k.lemma.token.txt', 'Flickr_8k.testImages.txt', 'features.pkl', 'Flicker8k_Dataset', 'Flickr8k_text.zip.1', 'Flickr8k_Dataset.zip', 'sample_data']


In [2]:
import os
import numpy as np
from tqdm import tqdm
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
import pickle

model = VGG16()
model = Model(inputs=model.inputs, outputs=model.layers[-2].output)

features = {}
folder = "Flicker8k_Dataset"

for img_name in tqdm(os.listdir(folder)):
    path = os.path.join(folder, img_name)

    img = load_img(path, target_size=(224,224))
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)

    feature = model.predict(img, verbose=0)

    image_id = img_name.split('.')[0]
    features[image_id] = feature

pickle.dump(features, open("features.pkl", "wb"))

print("DONE FEATURES:", len(features))

100%|██████████| 8091/8091 [14:18<00:00,  9.43it/s]


DONE FEATURES: 8091


In [3]:
def load_desc(file):
    descriptions = {}
    with open(file, 'r') as f:
        for line in f:
            tokens = line.split()
            if len(tokens) < 2:
                continue

            img_id = tokens[0].split('.')[0]
            caption = ' '.join(tokens[1:])
            caption = "startseq " + caption + " endseq"

            if img_id not in descriptions:
                descriptions[img_id] = []

            descriptions[img_id].append(caption)

    return descriptions

descriptions = load_desc("Flickr8k.token.txt")
print("Descriptions:", len(descriptions))

Descriptions: 8092


In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer

all_caps = []
for key in descriptions:
    all_caps.extend(descriptions[key])

tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_caps)

vocab_size = len(tokenizer.word_index) + 1
print(vocab_size)

8496


In [5]:
max_length = max(len(c.split()) for c in all_caps)
print(max_length)

40


In [6]:
features = pickle.load(open("features.pkl", "rb"))

descriptions = {k: descriptions[k] for k in descriptions if k in features}

print("After filter:", len(descriptions))

After filter: 8091


In [9]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import numpy as np

def data_generator(descriptions, features, tokenizer, max_length, vocab_size, batch_size=32):
    while True:
        X1, X2, y = [], [], []

        for key, desc_list in descriptions.items():
            photo = features[key][0]

            for desc in desc_list:
                seq = tokenizer.texts_to_sequences([desc])[0]

                for i in range(1, len(seq)):
                    in_seq = seq[:i]
                    out_seq = seq[i]

                    in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                    out_seq = to_categorical(out_seq, num_classes=vocab_size)

                    X1.append(photo)
                    X2.append(in_seq)
                    y.append(out_seq)

                    # 🔥 yield batch
                    if len(X1) == batch_size:
                        yield ([np.array(X1), np.array(X2)], np.array(y))
                        X1, X2, y = [], [], []

In [10]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add

# image input
input1 = Input(shape=(4096,))
x1 = Dropout(0.5)(input1)
x1 = Dense(256, activation='relu')(x1)

# text input
input2 = Input(shape=(max_length,))
x2 = Embedding(vocab_size, 256, mask_zero=True)(input2)
x2 = Dropout(0.5)(x2)
x2 = LSTM(256)(x2)

# merge
x = add([x1, x2])
x = Dense(256, activation='relu')(x)
output = Dense(vocab_size, activation='softmax')(x)

model = Model(inputs=[input1, input2], outputs=output)

In [11]:
model.compile(loss='categorical_crossentropy', optimizer='adam')
print("MODEL READY ✅")

MODEL READY ✅


In [13]:
steps = len(descriptions) // 2   # reduce if slow

generator = data_generator(descriptions, features, tokenizer, max_length, vocab_size)

model.fit(generator, epochs=5, steps_per_epoch=steps, verbose=1)

TypeError: `output_signature` must contain objects that are subclass of `tf.TypeSpec` but found <class 'list'> which is not.

In [14]:
def data_generator(descriptions, features, tokenizer, max_length, vocab_size, batch_size=32):
    while True:
        X1, X2, y = [], [], []

        for key, desc_list in descriptions.items():
            photo = features[key][0]

            for desc in desc_list:
                seq = tokenizer.texts_to_sequences([desc])[0]

                for i in range(1, len(seq)):
                    in_seq = seq[:i]
                    out_seq = seq[i]

                    in_seq = pad_sequences([in_seq], maxlen=max_length, padding='post')[0]
                    out_seq = to_categorical(out_seq, num_classes=vocab_size)

                    X1.append(photo)
                    X2.append(in_seq)
                    y.append(out_seq)

                    if len(X1) == batch_size:
                        yield (np.array(X1), np.array(X2)), np.array(y)
                        X1, X2, y = [], [], []

In [15]:
steps = len(descriptions) // 2

generator = data_generator(descriptions, features, tokenizer, max_length, vocab_size)

model.fit(generator, epochs=5, steps_per_epoch=steps)

Epoch 1/5


InvalidArgumentError: Graph execution error:

Detected at node functional_1_1/lstm_1/Assert/Assert defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>

  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start

  File "/usr/local/lib/python3.12/dist-packages/tornado/platform/asyncio.py", line 211, in start

  File "/usr/lib/python3.12/asyncio/base_events.py", line 645, in run_forever

  File "/usr/lib/python3.12/asyncio/base_events.py", line 1999, in _run_once

  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run

  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py", line 510, in dispatch_queue

  File "/usr/local/lib/python3.1

In [16]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import numpy as np

def data_generator(descriptions, features, tokenizer, max_length, vocab_size, batch_size=32):
    while True:
        X1, X2, y = [], [], []

        for key, desc_list in descriptions.items():
            photo = features[key][0]

            for desc in desc_list:
                seq = tokenizer.texts_to_sequences([desc])[0]

                for i in range(1, len(seq)):
                    in_seq = seq[:i]
                    out_seq = seq[i]

                    # ✅ RIGHT PADDING (IMPORTANT)
                    in_seq = pad_sequences([in_seq], maxlen=max_length, padding='post')[0]

                    # ✅ ONE HOT
                    out_seq = to_categorical(out_seq, num_classes=vocab_size)

                    X1.append(photo)
                    X2.append(in_seq)
                    y.append(out_seq)

                    if len(X1) == batch_size:
                        yield (np.array(X1), np.array(X2)), np.array(y)
                        X1, X2, y = [], [], []

In [17]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add

# IMAGE FEATURE EXTRACTOR
inputs1 = Input(shape=(2048,))
fe1 = Dropout(0.5)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)

# TEXT SEQUENCE MODEL
inputs2 = Input(shape=(max_length,))
se1 = Embedding(vocab_size, 256, mask_zero=False)(inputs2)  # ✅ mask_zero=False
se2 = Dropout(0.5)(se1)
se3 = LSTM(256)(se2)

# DECODER
decoder1 = add([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)

# FINAL MODEL
model = Model(inputs=[inputs1, inputs2], outputs=outputs)
model.compile(loss='categorical_crossentropy', optimizer='adam')

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer         │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 40, 256)   │  2,174,976 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 2048)      │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 40, 256)   │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    524,544 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 256)       │    525,312 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 256)       │          0 │ dense[0][0],      │
│                     │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │     65,792 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 8496)      │  2,183,472 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,474,096 (20.88 MB)

 Trainable params: 5,474,096 (20.88 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
# steps calculation
steps = len(descriptions) // 32

# generator
generator = data_generator(descriptions, features, tokenizer, max_length, vocab_size)

# TRAIN
model.fit(generator, epochs=5, steps_per_epoch=steps, verbose=1)

Epoch 1/5


ValueError: Input 0 with name 'input_layer' of layer 'functional' is incompatible with the layer: expected shape=(None, 2048), found shape=(None, 4096)

In [19]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add

# ✅ CHANGE HERE (2048 → 4096)
inputs1 = Input(shape=(4096,))
fe1 = Dropout(0.5)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)

# TEXT MODEL
inputs2 = Input(shape=(max_length,))
se1 = Embedding(vocab_size, 256, mask_zero=False)(inputs2)
se2 = Dropout(0.5)(se1)
se3 = LSTM(256)(se2)

# DECODER
decoder1 = add([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)

model = Model(inputs=[inputs1, inputs2], outputs=outputs)
model.compile(loss='categorical_crossentropy', optimizer='adam')

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 4096)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 40, 256)   │  2,174,976 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 4096)      │          0 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 40, 256)   │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 256)       │  1,048,832 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 256)       │    525,312 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 256)       │          0 │ dense_3[0][0],    │
│                     │                   │            │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 256)       │     65,792 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 8496)      │  2,183,472 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,998,384 (22.88 MB)

 Trainable params: 5,998,384 (22.88 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
model.fit(generator, epochs=5, steps_per_epoch=steps)

Epoch 1/5
252/252 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 6.0481
Epoch 2/5
252/252 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 5.6388
Epoch 3/5
252/252 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 5.6516
Epoch 4/5
252/252 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 5.5184
Epoch 5/5
252/252 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 5.4966


In [21]:
from tensorflow.keras.models import load_model
import pickle

model = load_model("model.h5")
tokenizer = pickle.load(open("tokenizer.pkl", "rb"))

print("✅ Model Loaded")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [22]:
model.save("model.h5")
print("✅ Model Saved")

✅ Model Saved


In [23]:
import pickle
pickle.dump(tokenizer, open("tokenizer.pkl", "wb"))
print("✅ Tokenizer Saved")

✅ Tokenizer Saved


In [24]:
import os
print(os.listdir())

['.config', 'Flickr8k_Dataset.zip.1', '__MACOSX', 'Flickr_8k.trainImages.txt', 'Flickr8k.token.txt', 'CrowdFlowerAnnotations.txt', 'tokenizer.pkl', 'ExpertAnnotations.txt', 'readme.txt', 'Flickr8k_text.zip', 'Flickr_8k.devImages.txt', 'Flickr8k.lemma.token.txt', 'Flickr_8k.testImages.txt', 'features.pkl', 'Flicker8k_Dataset', 'Flickr8k_text.zip.1', 'Flickr8k_Dataset.zip', 'model.h5', 'sample_data']


In [25]:
from tensorflow.keras.models import load_model
import pickle

model = load_model("model.h5")
tokenizer = pickle.load(open("tokenizer.pkl", "rb"))

print("✅ Model Loaded")

✅ Model Loaded


In [26]:
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
import numpy as np

# load model
base_model = VGG16()
model_vgg = Model(inputs=base_model.inputs, outputs=base_model.layers[-2].output)

def extract_feature(img_path):
    img = load_img(img_path, target_size=(224,224))
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)

    feature = model_vgg.predict(img, verbose=0)
    return feature

In [27]:
def idx_to_word(integer, tokenizer):
    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word
    return None

In [28]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def generate_caption(model, tokenizer, photo, max_length):
    in_text = "startseq"

    for i in range(max_length):
        seq = tokenizer.texts_to_sequences([in_text])[0]
        seq = pad_sequences([seq], maxlen=max_length, padding='post')

        yhat = model.predict([photo, seq], verbose=0)
        yhat = np.argmax(yhat)

        word = idx_to_word(yhat, tokenizer)

        if word is None:
            break

        in_text += " " + word

        if word == "endseq":
            break

    return in_text

In [29]:
img_path = "Flicker8k_Dataset/1000268201_693b08cb0e.jpg"

photo = extract_feature(img_path)

caption = generate_caption(model, tokenizer, photo, max_length)

print("🔥 Caption:", caption)

🔥 Caption: startseq a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a


In [32]:
from tensorflow.keras.optimizers import Adam

# Recompile with fresh optimizer
model.compile(loss='categorical_crossentropy', optimizer=Adam())

# Train again
model.fit(generator, epochs=20, steps_per_epoch=steps)

Epoch 1/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 5.4144
Epoch 2/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 5.5132
Epoch 3/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - loss: 5.4252
Epoch 4/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 5.3934
Epoch 5/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 5.4054
Epoch 6/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 5.2849
Epoch 7/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 5.2674
Epoch 8/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 5.3124
Epoch 9/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 5.4323
Epoch 10/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 5.3587
Epoch 11/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 5.3020
Epoch 12/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 5.2522
Epoch 13/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 5.2941
Epoch 14/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 5.3590
Epoch 15/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 3s

In [34]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

def generate_caption_beam_search(model, tokenizer, photo, max_length, beam_index=3):
    start = [tokenizer.word_index['startseq']]

    sequences = [[start, 0.0]]

    while len(sequences[0][0]) < max_length:
        all_candidates = []

        for seq, score in sequences:
            padded = pad_sequences([seq], maxlen=max_length, padding='post')
            preds = model.predict([photo, padded], verbose=0)[0]

            # get top beam_index predictions
            top_preds = np.argsort(preds)[-beam_index:]

            for word in top_preds:
                next_seq = seq + [word]
                next_score = score - np.log(preds[word] + 1e-10)
                all_candidates.append([next_seq, next_score])

        # sort and select best
        ordered = sorted(all_candidates, key=lambda tup: tup[1])
        sequences = ordered[:beam_index]

    # best sequence
    best_seq = sequences[0][0]

    # convert to words
    caption = []
    for i in best_seq:
        word = None
        for w, index in tokenizer.word_index.items():
            if index == i:
                word = w
                break

        if word == 'endseq':
            break

        if word != 'startseq':
            caption.append(word)

    return ' '.join(caption)

In [35]:
img_path = "Flicker8k_Dataset/1000268201_693b08cb0e.jpg"

photo = extract_feature(img_path)

caption = generate_caption_beam_search(model, tokenizer, photo, max_length)

print("🔥 Final Caption:", caption)

🔥 Final Caption: a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a


In [36]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model_blip = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

image = Image.open(img_path)

inputs = processor(image, return_tensors="pt")
out = model_blip.generate(**inputs)

print(processor.decode(out[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

a little girl in a pink dress


In [40]:

img_path = "Flicker8k_Dataset/33108590_d685bfe51c.jpg"

photo = extract_feature(img_path)

caption = generate_caption_beam_search(model, tokenizer, photo, max_length)

print("🔥 Final Caption:", caption)

🔥 Final Caption: a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a


In [41]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model_blip = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

image = Image.open(img_path)

inputs = processor(image, return_tensors="pt")
out = model_blip.generate(**inputs)

print(processor.decode(out[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

a woman standing in front of a car


In [42]:
pip install streamlit pillow transformers torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 126.6 MB/s eta 0:00:00


In [44]:
%%writefile app.py
import streamlit as st
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

# Load model
@st.cache_resource
def load_model():
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
    return processor, model

processor, model = load_model()

st.title("🧠 Image Caption Generator")
st.write("Upload an image and get caption")

uploaded_file = st.file_uploader("Upload Image", type=["jpg","png","jpeg"])

if uploaded_file:
    image = Image.open(uploaded_file).convert('RGB')
    st.image(image, caption="Uploaded Image", use_column_width=True)

    if st.button("Generate Caption"):
        with st.spinner("Generating..."):
            inputs = processor(image, return_tensors="pt")
            out = model.generate(**inputs)
            caption = processor.decode(out[0], skip_special_tokens=True)

        st.success("Caption Generated!")
        st.write(caption)

Writing app.py


In [45]:
!pip install streamlit pyngrok transformers pillow torch torchvision

In [47]:
import streamlit as st
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

# Load model (only once)
@st.cache_resource
def load_model():
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
    return processor, model

processor, model = load_model()

# UI
st.set_page_config(page_title="Image Caption Generator", page_icon="🧠")

st.title("🧠 AI Image Caption Generator")
st.write("Upload an image and get a smart caption 🔥")

# Upload image
uploaded_file = st.file_uploader("📤 Upload Image", type=["jpg", "png", "jpeg"])

if uploaded_file is not None:
    image = Image.open(uploaded_file).convert('RGB')

    st.image(image, caption="Uploaded Image", use_column_width=True)

    if st.button("🚀 Generate Caption"):
        with st.spinner("Generating caption..."):
            inputs = processor(image, return_tensors="pt")
            out = model.generate(**inputs)
            caption = processor.decode(out[0], skip_special_tokens=True)

        st.success("✅ Caption Generated!")
        st.subheader("📝 Caption:")
        st.write(caption)

2026-03-22 19:11:52.072 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

In [49]:
from pyngrok import ngrok

# kill previous tunnels
ngrok.kill()

# run streamlit in background
!streamlit run app.py &>/dev/null&

# create public URL
public_url = ngrok.connect(8501)
print("🔥 Open this link:", public_url)

ERROR:pyngrok.process.ngrok:t=2026-03-22T19:13:14+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-03-22T19:13:14+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-03-22T19:13:14+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [50]:
!ngrok config add-authtoken 3BJUyYhqvDUTOHRuQhB0fRAmO5p_5HcKhvFDQrMW8sDnKuWEh

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [51]:
from pyngrok import ngrok

ngrok.kill()

!streamlit run app.py &>/dev/null&

public_url = ngrok.connect(8501)
print("🔥 Open this link:", public_url)

🔥 Open this link: NgrokTunnel: "https://reborn-boneless-zulma.ngrok-free.dev" -> "http://localhost:8501"
